# Optional hosted classification: collect separately, evaluate offline
This notebook makes **zero API requests**. Read the E4 README before a separate provider run. Freeze the 80 test IDs, label policy, prompt/model revision, output-token cap, and call/retry limit. Record refusals/errors/missing outputs, real token usage/cost/latency only when measured, and retain source IDs. Store the collected envelope at repository `.cache/hosted-e4-results.json`. Never put credentials in this notebook or the static site.
Required envelope fields: provider, model, prompt_version, collected_at, outputs. Each output has id, raw (plain label or JSON string with exactly label), optional error/latency_seconds. This importer rejects unknown/duplicate IDs and includes missing/error outputs as invalid predictions. External provider execution is not verified by the atlas.


In [ ]:
import sys, json
from pathlib import Path
project = Path.cwd()
if project.name == 'notebooks': project = project.parent
root = project.parents[2]
sys.path.insert(0, str(root / 'examples/foundations/src'))
from atlas_foundations.e4.hosted_results import evaluate_external
fixture = json.loads((root / 'examples/data/fixtures/sentiment.json').read_text())
collected = root / '.cache/hosted-e4-results.json'
if collected.exists():
    report = evaluate_external(fixture, json.loads(collected.read_text()))
    print(json.dumps(report, indent=2))
else:
    print('No hosted results supplied; no API was called and no hosted metrics are claimed.')
    print('Required test IDs:', [r['id'] for r in fixture['rows'] if r['split'] == 'test'])
